In [2]:
import pandas as pd

# Loading original files
all_matches = pd.read_csv('all_matches.csv')
countries_names = pd.read_csv('countries_names.csv')

# 2. Map historical names to modern names
name_mapping = dict(zip(countries_names['original_name'], countries_names['current_name']))
all_matches['home_team'] = all_matches['home_team'].map(lambda x: name_mapping.get(x, x))
all_matches['away_team'] = all_matches['away_team'].map(lambda x: name_mapping.get(x, x))
all_matches['date'] = pd.to_datetime(all_matches['date'])

# 3. Create two rows for every match (Team vs Opponent perspective)
home_perspective = all_matches.copy().rename(columns={
    'home_team': 'team', 'away_team': 'opponent',
    'home_score': 'gf', 'away_score': 'ga'
})
home_perspective['venue'] = home_perspective['neutral'].map({True: 'Neutral', False: 'Home'})

away_perspective = all_matches.copy().rename(columns={
    'away_team': 'team', 'home_team': 'opponent',
    'away_score': 'gf', 'home_score': 'ga'
})
away_perspective['venue'] = away_perspective['neutral'].map({True: 'Neutral', False: 'Away'})

# 4. Combine and add ML codes (venue_code, opp_code, day_code)
matches_transformed = pd.concat([home_perspective, away_perspective], ignore_index=True)
matches_transformed['target'] = (matches_transformed['gf'] > matches_transformed['ga']).astype(int)
matches_transformed['venue_code'] = matches_transformed['venue'].astype('category').cat.codes
matches_transformed['opp_code'] = matches_transformed['opponent'].astype('category').cat.codes
matches_transformed['day_code'] = matches_transformed['date'].dt.dayofweek

# 5. Save the file and download it to your computer
matches_transformed.to_csv('international_matches_transformed.csv', index=False)

from google.colab import files
files.download('international_matches_transformed.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>